In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

#Load Data

df = pd.read_csv('../data/raw/train.csv')
print("Original Data Shape:", df.shape)

Original Data Shape: (1460, 81)


In [2]:
#Remove Outliers found in EDA
#GrLivArea > 4000 with low price are suspicious


df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))]
print("After Outlier Removal Data Shape:", df.shape)

After Outlier Removal Data Shape: (1458, 81)


In [3]:
#Log Transform target variable (fix skewness)

y = np.log1p(df['SalePrice'])
X = df.drop(['SalePrice'], axis=1)

print("Target (Y) Variable Shape:", y.shape)
print("Feature (X) Variable Shape:", X.shape)

Target (Y) Variable Shape: (1458,)
Feature (X) Variable Shape: (1458, 80)


In [5]:
#Check missing values in features

missing = X.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Missing Values in Features: {len(missing)}\n")
print(missing)

Missing Values in Features: 19

PoolQC          1452
MiscFeature     1404
Alley           1367
Fence           1177
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64


In [6]:
#Separate numerical and categorical columns

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical Columns: {len(numerical_cols)}")
print(f"Categorical Columns: {len(categorical_cols)}")

Numerical Columns: 37
Categorical Columns: 43


/var/folders/mn/nxx3gsxx4xs3bqw2f579j6b40000gn/T/ipykernel_23282/656463639.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [7]:
#Fill missing values for numerical columns with median
#Categorical : "None" means the feature does not exist (e.g. no garage, no pool)


for col in categorical_cols:
    X[col] = X[col].fillna("None")

#Numerical : Fill with median (resistant to outliers)
for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())

#Verify no missing values left
print("Remaining missing values:", X.isnull().sum().sum())

Remaining missing values: 0


In [8]:
# ---- FEATURE ENGINEERING ----

# Total living area (basement + 1st floor + 2nd floor)
X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]

# Total bathrooms (full + half counted as 0.5)
X["TotalBath"] = (X["FullBath"] + X["BsmtFullBath"] +
                  0.5 * X["HalfBath"] + 0.5 * X["BsmtHalfBath"])

# Total porch area
X["TotalPorchSF"] = (X["OpenPorchSF"] + X["EnclosedPorch"] +
                     X["3SsnPorch"] + X["ScreenPorch"])

# House age at time of sale
X["HouseAge"] = X["YrSold"] - X["YearBuilt"]

# Remodel age at time of sale
X["RemodAge"] = X["YrSold"] - X["YearRemodAdd"]

# Was the house remodeled? (binary)
X["IsRemodeled"] = (X["YearBuilt"] != X["YearRemodAdd"]).astype(int)

# Binary flags
X["HasPool"] = (X["PoolArea"] > 0).astype(int)
X["HasGarage"] = (X["GarageArea"] > 0).astype(int)
X["Has2ndFloor"] = (X["2ndFlrSF"] > 0).astype(int)
X["HasFireplace"] = (X["Fireplaces"] > 0).astype(int)

# Quality score (interaction feature)
X["OverallScore"] = X["OverallQual"] * X["OverallCond"]

print("New shape after feature engineering:", X.shape)
print("\nNew features added:")
new_features = ["TotalSF", "TotalBath", "TotalPorchSF", "HouseAge",
                "RemodAge", "IsRemodeled", "HasPool", "HasGarage",
                "Has2ndFloor", "HasFireplace", "OverallScore"]
for f in new_features:
    print(f"  {f}: {X[f].describe().round(2).to_dict()}")

New shape after feature engineering: (1458, 91)

New features added:
  TotalSF: {'count': 1458.0, 'mean': 2557.15, 'std': 774.11, 'min': 334.0, '25%': 2008.5, '50%': 2473.0, '75%': 3002.25, 'max': 6872.0}
  TotalBath: {'count': 1458.0, 'mean': 2.21, 'std': 0.78, 'min': 1.0, '25%': 2.0, '50%': 2.0, '75%': 2.5, 'max': 6.0}
  TotalPorchSF: {'count': 1458.0, 'mean': 86.73, 'std': 104.79, 'min': 0.0, '25%': 0.0, '50%': 48.0, '75%': 135.75, 'max': 1027.0}
  HouseAge: {'count': 1458.0, 'mean': 36.6, 'std': 30.24, 'min': 0.0, '25%': 8.0, '50%': 35.0, '75%': 54.0, 'max': 136.0}
  RemodAge: {'count': 1458.0, 'mean': 22.98, 'std': 20.64, 'min': 0.0, '25%': 4.0, '50%': 14.0, '75%': 41.0, 'max': 60.0}
  IsRemodeled: {'count': 1458.0, 'mean': 0.48, 'std': 0.5, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 1.0}
  HasPool: {'count': 1458.0, 'mean': 0.0, 'std': 0.06, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}
  HasGarage: {'count': 1458.0, 'mean': 0.94, 'std': 0.23, 'min': 0.0

In [9]:
#Encode categorical variables using one-hot encoding
X_encoded = pd.get_dummies(X, drop_first=True)

print("Shape before encoding:", X.shape)
print("Shape after encoding:", X_encoded.shape)
print(f"\n{X_encoded.shape[1] - X.shape[1]} new columns created from categorical variables")


Shape before encoding: (1458, 91)
Shape after encoding: (1458, 271)

180 new columns created from categorical variables


In [10]:
#Fix skewed numerical features

from scipy.stats import skew

numeric_cols = X_encoded.select_dtypes(include=["int64", "float64"]).columns
skewed = X_encoded[numeric_cols].apply(skew).sort_values(ascending=False)
high_skew = skewed[abs(skewed) > 0.75].index
print(f"Skewed features (|skew| > 0.75): {len(high_skew)}")
X_encoded[high_skew] = np.log1p(X_encoded[high_skew])
print("Log transform applied to skewed features")

Skewed features (|skew| > 0.75): 24
Log transform applied to skewed features


In [11]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,       # %20 test, %80 train
    random_state=42       # Reproducibility (her seferinde ayni bolme)
)

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")
print(f"\nTrain/Test ratio: {len(X_train)/len(X_encoded)*100:.0f}% / {len(X_test)/len(X_encoded)*100:.0f}%")

Training set: (1166, 271)
Testing set:  (292, 271)

Train/Test ratio: 80% / 20%
